# Hyperparameter tuning of PoseRFM

A search over the hyperparameters of PoseRFM (`hp/models/poserfm.py`, Riemannian flow matching on SO(3)^19) with [Optuna](https://optuna.org), on every 10th frame of AMASS (`~/data/AMASS_rotations_c10`, official split): the data of the best RSGM (trial 34 of its search, 139.1-139.2 nats on the test split), so that the two can be compared. The defaults (`config/models/poserfm.yaml`) are the settings of PoseD-Flow, except that the step schedule stands in for ReduceLROnPlateau.

**The objective** is the mean log-likelihood of a fixed, stratified sample of validation poses (`N_PER_DATASET` of each valid dataset; EMA weights; `OBJECTIVE_STEPS` ODE steps), after `TRIAL_ITERATIONS` training steps: the same objective as the search of RSGM (`rsgm_hyperparameter_tuning.ipynb`), so the two searches compare. Not the flow matching loss: it depends on how the times are drawn, so it does not compare between configurations. The test split is not used here.

**Budget.** 20k steps per trial: on every 10th frame the validation loss of PoseRFM gains most of what it gains in 50k steps in the first 20k. At batch 4096 a trial takes ~7.5 min of training on mps plus ~1 min of likelihoods.

**The search.** TPE sampler (multivariate, with `group=True` for the conditional parameter `time_mean`), the first 10 trials random, the first of them the configuration of the paper. A median pruner checks the log-likelihood (half of the validation poses, `PRUNE_STEPS` ODE steps) every `PRUNE_EVERY` steps. The study lives in SQLite (`STUDY_DIR/optuna.db`): the notebook can be stopped and run again, it continues where it stopped, and `optuna-dashboard sqlite:///$HOME/data/experiments/poserfm_hpo/optuna.db` shows it live.

**Noise.** The noise between training seeds of RSGM is ~0.2 nats: differences of a few tenths between the best trials are no signal. The final retraining (optional, `RUN_FINAL`) trains the best configurations at the full budget with two seeds.

In [ ]:
import json
import logging
import os
import time
from pathlib import Path
from unittest import mock

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from omegaconf import OmegaConf

from hp.data.amass import AMASS
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS, AmassJoints4Learning
from hp.models import build_model
from hp.models.io import load_model, save_model
from hp.training import TrainConfig, Trainer, cli

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_c10"      # training and validation data: every 10th frame
SPLIT = "original"                            # the official AMASS split
STUDY_DIR = "~/data/experiments/poserfm_hpo"  # the Optuna database and the weights of the trials
DEVICE = "mps"                                # "cpu", "mps" or "cuda:0"
SEED = 10
TRIAL_ITERATIONS = 20_000                     # training steps of a trial
PRUNE_EVERY = 5_000                           # steps between the checks of the pruner
BATCH_SIZE = 4096                             # fixed: at a fixed number of steps a larger batch only sees more data
N_PER_DATASET = 500                           # validation poses per valid dataset (4 datasets)
OBJECTIVE_STEPS = 200                         # ODE steps of the objective's log-likelihood
PRUNE_STEPS = 100                             # ODE steps of the pruner's checks (on half of the poses)
WARMUP_STEPS = 500                            # linear warm-up of the cosine schedule

STUDY_NAME, N_TRIALS = "poserfm_c10", 40

RUN_FINAL = False                             # True: retrain the best configurations at the full budget (hours)
TOP_K = 3                                     # best configurations retrained (plus the paper's)
FINAL_ROOT = "~/data/experiments/poserfm_tuned"  # their runs: FINAL_ROOT/<config>/run_0001, ...
FINAL_ITERATIONS = 50_000
FINAL_SEEDS = [10, 11]
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
STUDY_DIR, FINAL_ROOT = expand(STUDY_DIR), expand(FINAL_ROOT)
STUDY_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{STUDY_DIR / 'optuna.db'}"
States = optuna.trial.TrialState
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)

# the settings of the paper (config/models/poserfm.yaml), in the terms of the search space
DEFAULT = dict(lr=1e-3, width=512, depth=4, activation="silu", weight_decay=1e-4, ema_decay=0.99, scheduler="step",
               time_sampling="uniform")

## 1. Data

The training split is loaded once and shared by all trials. The validation poses: `N_PER_DATASET` random poses of every valid dataset (HumanEva, HDM05, SFU, MoSh), drawn with a fixed seed. Without the stratification HDM05 (79% of the split) would dominate. The first half of each dataset's poses is the subset of the pruner's checks.

In [ ]:
with mock.patch.dict(os.environ, {"HUPROSO_AMASS_SPLIT": SPLIT}):
    train_set = AMASS(mode="train", path_to_dataset=ROTATIONS, rotation_representation="matrix",
                      selected_joints=AmassJoints4Learning, load_dtype="float32")

rng = np.random.default_rng(SEED)
parts, half = [], []
for name in ORIGINAL_AMASS_SPLITS["valid"]:
    poses = AMASS(datasets=[name], path_to_dataset=ROTATIONS, rotation_representation="matrix",
                  selected_joints=AmassJoints4Learning).data.reshape(-1, len(AmassJoints4Learning), 3, 3)
    chosen = poses[rng.choice(len(poses), min(N_PER_DATASET, len(poses)), replace=False)]
    parts.append(chosen)
    half.append(chosen[: len(chosen) // 2])
    print(f"valid {name:9s}: {len(poses):7,} poses, {len(chosen)} chosen")
valid_poses, prune_poses = torch.cat(parts).float(), torch.cat(half).float()
print(f"train: {len(train_set):,} poses; validation poses: {len(valid_poses):,} (objective), {len(prune_poses):,} (pruner)")


@torch.no_grad()
def log_likelihood(model, poses, steps, batch=250):
    # Log-likelihoods [N] of the poses under the model, with `steps` ODE steps.
    old, model.likelihood_steps = model.likelihood_steps, steps
    try:
        return torch.cat([model.log_prob(poses[i:i + batch].to(model.device)).float().cpu() for i in range(0, len(poses), batch)])
    finally:
        model.likelihood_steps = old

## 2. Search space and the objective

| parameter | range | paper | why |
|---|---|---|---|
| `lr` | 1e-4 ... 5e-3 (log) | 1e-3 | the main lever of RSGM's search |
| `width`, `depth` of the MLP | 256 / 512 / 1024, 3 ... 6 | 512, 4 | capacity: the other main lever of RSGM's search |
| `activation` | silu, sin | silu | |
| `weight_decay` (AdamW) | 1e-6 ... 1e-2 (log) | 1e-4 | from almost none to PyTorch's default |
| `ema_decay` | 0.99, 0.995, 0.999 | 0.99 | |
| `scheduler` | step, cosine (with warm-up) | ReduceLROnPlateau | plateau is not in the trainer; cosine is the usual choice for a fixed budget |
| `time_sampling` | uniform, logit_normal | uniform | the times of the loss: logit-normal puts weight near the data (t → 1), where the likelihood accumulates most of its divergence |
| `time_mean` (only for logit_normal) | 0 ... 1.5 | – | 0: centred, larger: towards the data; `time_std` fixed at 1 |

Fixed: the batch (4096, as the paper), `eps` = 1e-3 (it defines what log p means), the number of ODE steps of the objective, gradient clipping 1.0, AdamW.

A trial builds the model, trains it with `Trainer`, reports the pruner's log-likelihood every `PRUNE_EVERY` steps, and returns the mean log-likelihood of the validation poses (EMA weights). A trial whose loss or likelihood stops being finite is pruned. The EMA weights of a finished trial are saved under `STUDY_DIR/trials`.

In [ ]:
def split_params(p):
    # A dict of search parameters -> (arguments of the model, fields of TrainConfig).
    model_args = dict(hidden=[p["width"]] * p["depth"], activation=p["activation"], time_sampling=p["time_sampling"],
                      likelihood_steps=OBJECTIVE_STEPS)
    if p["time_sampling"] == "logit_normal":
        model_args.update(time_mean=p["time_mean"], time_std=1.0)
    train = dict(lr=p["lr"], batch_size=BATCH_SIZE, optimizer="adamw", weight_decay=p["weight_decay"], ema_decay=p["ema_decay"],
                 scheduler=p["scheduler"], warmup_steps=WARMUP_STEPS if p["scheduler"] == "cosine" else 0)
    return model_args, train


def suggest(trial):
    p = dict(
        lr=trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        width=trial.suggest_categorical("width", [256, 512, 1024]),
        depth=trial.suggest_int("depth", 3, 6),
        activation=trial.suggest_categorical("activation", ["silu", "sin"]),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        ema_decay=trial.suggest_categorical("ema_decay", [0.99, 0.995, 0.999]),
        scheduler=trial.suggest_categorical("scheduler", ["step", "cosine"]),
        time_sampling=trial.suggest_categorical("time_sampling", ["uniform", "logit_normal"]),
    )
    if p["time_sampling"] == "logit_normal":
        p["time_mean"] = trial.suggest_float("time_mean", 0.0, 1.5)
    return split_params(p)


def objective(trial, weights_dir):
    model_args, train = suggest(trial)
    torch.manual_seed(SEED + trial.number)
    model = build_model("poserfm", num_joints=len(AmassJoints4Learning), **model_args)
    config = TrainConfig(max_iterations=TRIAL_ITERATIONS, valid_interval=PRUNE_EVERY, checkpoint_every=0,
                         device=DEVICE, progress=False, **train)
    trainer = Trainer(model, train_set, None, config)

    def check(step, _):
        if step >= TRIAL_ITERATIONS:
            return  # the end: the objective below
        if not np.isfinite(trainer.history.loss[-1]):
            raise optuna.TrialPruned(f"non-finite loss at step {step}")
        ll = log_likelihood(trainer.ema.model, prune_poses, PRUNE_STEPS)
        value = float(ll.mean()) if torch.isfinite(ll).all() else -np.inf
        trial.report(value, step)
        print(f"  trial {trial.number:3d} step {step:6,}: log-likelihood {value:8.2f}", flush=True)
        if not np.isfinite(value) or trial.should_prune():
            raise optuna.TrialPruned(f"log-likelihood {value:.2f} at step {step}")

    start = time.perf_counter()
    try:
        trainer.fit(callback=check)
        ema = trainer.ema.model
        ll = log_likelihood(ema, valid_poses, OBJECTIVE_STEPS)
        if not torch.isfinite(ll).all():
            raise optuna.TrialPruned("non-finite log-likelihood")
        trial.set_user_attr("parameters", model.num_parameters())
        trial.set_user_attr("sem", float(ll.std() / np.sqrt(len(ll))))
        save_model(ema, Path(weights_dir) / f"trial_{trial.number:04d}")
        return float(ll.mean())
    finally:
        trial.set_user_attr("seconds", time.perf_counter() - start)
        del trainer, model
        if DEVICE == "mps":
            torch.mps.empty_cache()


def report(study, trial):
    # Optuna callback: one line per finished trial.
    value = f"{trial.value:.3f}" if trial.value is not None else "-"
    best = f"{study.best_value:.3f} (trial {study.best_trial.number})" if any(t.state == States.COMPLETE for t in study.trials) else "-"
    print(f"trial {trial.number:3d} {trial.state.name:8s} {trial.user_attrs.get('seconds', 0) / 60:5.1f} min, "
          f"log-likelihood {value}, best {best}", flush=True)


def run_study(study, n_trials, weights_dir):
    # Run the trials still missing to reach n_trials finished (complete or pruned) ones.
    done = [t for t in study.trials if t.state in (States.COMPLETE, States.PRUNED)]
    missing = max(0, n_trials - len(done))
    print(f"{study.study_name}: {len(done)} trials done, {missing} to run")
    start = time.perf_counter()
    study.optimize(lambda trial: objective(trial, weights_dir), n_trials=missing, callbacks=[report], catch=(RuntimeError,))
    print(f"{study.study_name}: {(time.perf_counter() - start) / 3600:.1f} h in this run")


def trials_table(study, extra=()):
    # All trials of a study, best first.
    rows = [{"number": t.number, "state": t.state.name, "value": t.value, **t.params, **{k: t.user_attrs.get(k) for k in extra},
             "params (M)": (t.user_attrs.get("parameters") or float("nan")) / 1e6, "minutes": t.user_attrs.get("seconds", float("nan")) / 60}
            for t in study.trials]
    return pd.DataFrame(rows).sort_values("value", ascending=False)


pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

## 3. The search

The first trial is the configuration of the paper (the reference), the next nine random (TPE's start-up).

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME, storage=STORAGE, load_if_exists=True, direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=0),
)
if not study.trials:
    study.enqueue_trial(DEFAULT)
run_study(study, N_TRIALS, STUDY_DIR / "trials")

## 4. Results

In [ ]:
df = trials_table(study, extra=("sem",))
print(df.head(15).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print(f"\nstates: {df.state.value_counts().to_dict()}; the configuration of the paper (trial 0): {study.trials[0].value}")

complete_df = df[df.state == "COMPLETE"]
print("\nmean / max objective by value of the categorical parameters:")
for p in ["width", "depth", "activation", "ema_decay", "scheduler", "time_sampling"]:
    if p in complete_df:
        stats = complete_df.groupby(p)["value"].agg(["count", "mean", "max"]).round(2)
        print(f"  {p:13s} " + "; ".join(f"{k}: n={r['count']:.0f} mean {r['mean']:.1f} max {r['max']:.1f}" for k, r in stats.iterrows()))

In [ ]:
complete = [t for t in study.trials if t.state == States.COMPLETE]
if complete:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot([t.number for t in complete], [t.value for t in complete], "o", label="trial")
    ax.plot([t.number for t in complete], np.maximum.accumulate([t.value for t in complete]), "-", label="best so far")
    if study.trials[0].value is not None:
        ax.axhline(study.trials[0].value, color="0.5", ls="--", label="configuration of the paper")
    ax.set_xlabel("trial")
    ax.set_ylabel("validation log-likelihood")
    ax.legend()
    plt.show()

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.4), sharey=True)
    for ax, name in zip(axes, ["lr", "weight_decay", "width", "time_mean"]):
        ts = [t for t in complete if name in t.params]
        ax.scatter([t.params[name] for t in ts], [t.value for t in ts], c=[t.params["depth"] for t in ts], cmap="viridis")
        if name in ("lr", "weight_decay"):
            ax.set_xscale("log")
        ax.set_xlabel(name)
    axes[0].set_ylabel("validation log-likelihood (colour: depth)")
    values = [t.value for t in complete]
    axes[0].set_ylim(np.percentile(values, 10) - 1, max(values) + 0.5)
    fig.tight_layout()
    plt.show()
    # fANOVA importances are unreliable with a few dozen correlated trials: read them next to the grouped statistics.
    if len(complete) >= 10:
        print("fANOVA importances:", {k: round(v, 3) for k, v in optuna.importance.get_param_importances(study).items()})

## 5. The best configuration

As overrides for `poserfm_train.ipynb` (`OVERRIDES`), and how it differs from the paper.

In [ ]:
def as_overrides(params):
    model_args, train = split_params(params)
    out = [f"model.args.hidden={json.dumps(model_args['hidden'])}", f"model.args.activation={model_args['activation']}",
           f"model.args.time_sampling={model_args['time_sampling']}"]
    if model_args["time_sampling"] == "logit_normal":
        out += [f"model.args.time_mean={model_args['time_mean']}", "model.args.time_std=1.0"]
    return out + [f"train.lr={train['lr']}", f"train.batch_size={train['batch_size']}", "train.optimizer=adamw",
                  f"train.weight_decay={train['weight_decay']}", f"train.ema_decay={train['ema_decay']}",
                  f"train.scheduler={train['scheduler']}", f"train.warmup_steps={train['warmup_steps']}"]


if complete:
    best = study.best_trial
    print(f"best: trial {best.number}, validation log-likelihood {best.value:.2f} +- {best.user_attrs.get('sem', float('nan')):.2f} "
          f"after {TRIAL_ITERATIONS:,} steps (the paper's configuration: {study.trials[0].value})")
    print({k: (v, DEFAULT.get(k)) for k, v in best.params.items() if v != DEFAULT.get(k)}, "(best, paper)")
    print("for poserfm_train.ipynb:\nOVERRIDES = [" + ",\n             ".join(json.dumps(o) for o in as_overrides(best.params)) + "]")

## 6. Final retraining (optional)

With `RUN_FINAL = True`: the `TOP_K` best finished trials and the configuration of the paper, each trained `FINAL_ITERATIONS` steps with every seed of `FINAL_SEEDS` through `hp.training.cli` (ordinary runs of the model zoo in `FINAL_ROOT/<config>/run_<n>`, which `rsgm_huproso_comparison.ipynb` can evaluate), then evaluated on the same validation poses. The spread over the seeds is the noise a difference between configurations has to beat. Finished runs are not trained again. This takes hours (~19 min per run at the paper's size).

In [ ]:
def final_overrides(params, seed, label):
    return ["model=poserfm", f"run_root={FINAL_ROOT}", f"run_name={label}", f"seed={seed}", f"data.rotations={ROTATIONS}",
            f"data.split={SPLIT}", "data.load_dtype=float32", f"train.device={DEVICE}", f"train.max_iterations={FINAL_ITERATIONS}",
            "train.valid_interval=1000", "train.checkpoint_every=10000", "train.progress=false", "tensorboard=false",
            *as_overrides(params)]


def finished_runs(label):
    # seed -> run folder of the finished runs of a configuration
    runs = {}
    for run in sorted((FINAL_ROOT / label).glob("run_[0-9][0-9][0-9][0-9]")):
        status = run / "status.json"
        if status.is_file() and json.loads(status.read_text())["finished"]:
            runs[int(OmegaConf.load(run / "config.yaml").seed)] = run
    return runs


best_trials = sorted(complete, key=lambda t: t.value, reverse=True)[:TOP_K]
finalists = {"paper": DEFAULT, **{f"trial_{t.number:04d}": dict(t.params) for t in best_trials}}
if RUN_FINAL:
    for label, params in finalists.items():
        for seed in FINAL_SEEDS:
            if seed in finished_runs(label):
                print(f"{label} seed {seed}: already trained in {finished_runs(label)[seed]}")
                continue
            print(f"{label} seed {seed}: training {FINAL_ITERATIONS:,} steps ...", flush=True)
            start = time.perf_counter()
            run = cli.main(final_overrides(params, seed, label))
            (run / "train_time.json").write_text(json.dumps({"seconds": time.perf_counter() - start, "device": DEVICE}))
            print(f"  {(time.perf_counter() - start) / 60:.1f} min -> {run}", flush=True)
else:
    print("RUN_FINAL is False: no retraining. Finalists would be:", list(finalists))

In [ ]:
rows = []
for label, params in finalists.items():
    for seed, run in finished_runs(label).items():
        cache = run / "valid_hpo.json"
        if cache.is_file():
            value = json.loads(cache.read_text())["log_likelihood"]
        else:
            model = load_model(run, device=DEVICE, weights="weights_ema")
            value = float(log_likelihood(model, valid_poses, OBJECTIVE_STEPS).mean())
            cache.write_text(json.dumps({"log_likelihood": value, "steps": OBJECTIVE_STEPS, "poses": len(valid_poses)}))
        rows.append(dict(config=label, seed=seed, log_likelihood=value, run=str(run)))
if rows:
    final = pd.DataFrame(rows)
    summary = final.groupby("config")["log_likelihood"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
    print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
    winner = summary.index[0]
    print(f'\nbest: {winner}; for rsgm_huproso_comparison.ipynb: "PoseRFM tuned": dict(run="{FINAL_ROOT / winner}", weights="weights_ema")')
    (STUDY_DIR / "final.json").write_text(json.dumps({"finalists": finalists, "runs": rows}, indent=2))
else:
    print("no finished final runs")